# 🔍 相似图片检索可视化
滑动选择测试集编号，查看最相似的 5 张训练集图片及其标签。

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from ipywidgets import interact, IntSlider, Output
from IPython.display import display, HTML
import os
import warnings
warnings.filterwarnings('ignore', message='Glyph.*missing')

%matplotlib inline
plt.rcParams['figure.dpi'] = 150

In [2]:
# === 路径配置 ===
CSV_PATH = r"C:\Users\hp\Desktop\similarity_results.csv"
TRAIN_DIR = r"C:\Users\hp\PycharmProjects\DS\project\step2\train_images\train_images"
TEST_DIR  = r"C:\Users\hp\PycharmProjects\DS\project\step2\test_images\test_images"

# === 读取 CSV ===
df = pd.read_csv(CSV_PATH)

# 获取所有唯一的 query_id，按文件名排序
all_queries = sorted(df['query_id'].unique())
n_queries = len(all_queries)

print(f"✅ 测试集图片数: {n_queries}")
print(f"✅ 总匹配记录数: {len(df)}")
print(f"✅ 标签取值: {sorted(df['query_label'].unique())}")

✅ 测试集图片数: 194
✅ 总匹配记录数: 970
✅ 标签取值: [np.int64(0), np.int64(1)]


In [3]:
import io
from ipywidgets import interact, IntSlider
from IPython.display import Image

def load_image_safe(path):
    if os.path.exists(path):
        return mpimg.imread(path)
    return None

def show_similarity(idx):
    query_id = all_queries[idx]
    query_path = os.path.join(TEST_DIR, query_id)
    matches = df[df['query_id'] == query_id].sort_values('rank')

    fig, axes = plt.subplots(2, 3, figsize=(24, 14))
    axes = axes.flatten()

    # Query 图（左上角）
    img = load_image_safe(query_path)
    if img is not None:
        axes[0].imshow(img)
    axes[0].set_title(
        f"Query #{idx}\n{query_id}\nLabel: {matches.iloc[0]['query_label']}",
        fontsize=16, color='steelblue', fontweight='bold'
    )
    axes[0].axis('off')
    for spine in axes[0].spines.values():
        spine.set_edgecolor('steelblue')
        spine.set_linewidth(4)

    # Top-5 匹配
    for i, (_, row) in enumerate(matches.iterrows()):
        ax = axes[i + 1]
        match_img = load_image_safe(os.path.join(TRAIN_DIR, row['match_id']))
        if match_img is not None:
            ax.imshow(match_img)
        border_color = 'limegreen' if row['query_label'] == row['match_label'] else 'tomato'
        for spine in ax.spines.values():
            spine.set_edgecolor(border_color)
            spine.set_linewidth(4)
        ax.set_title(
            f"Top {i+1}  |  {row['match_id']}\nLabel: {row['match_label']}    sim: {row['similarity']:.4f}",
            fontsize=15, color=border_color
        )
        ax.axis('off')

    plt.tight_layout(pad=2.0)

    # 渲染到内存，关闭 fig，绕过 inline 自动显示
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=120, bbox_inches='tight')
    plt.close(fig)
    buf.seek(0)
    display(Image(data=buf.read()))

    correct_count = (matches['query_label'] == matches['match_label']).sum()
    query_lbl = matches.iloc[0]['query_label']
    display(HTML(
        f"<div style='font-size:17px; margin:8px 0;'>"
        f"<b>Query Label</b> = {query_lbl} &nbsp;|&nbsp; "
        f"<b>Top-5 正确数</b> = "
        f"<span style='color:{'green' if correct_count >= 3 else 'red'}; font-weight:bold'>"
        f"{correct_count}/5</span></div>"
    ))

interact(
    show_similarity,
    idx=IntSlider(
        min=0, max=n_queries - 1, step=1, value=0,
        description='测试集编号',
        continuous_update=False,
        style={'description_width': 'initial'},
        layout={'width': '80%'}
    )
);

interactive(children=(IntSlider(value=0, continuous_update=False, description='测试集编号', layout=Layout(width='80…